# Task #7: Clustering Experiments

**The job:** sort comments into themes — groups of comments about the same
thing — so the report stage has something to organize around instead of 112,000
loose comments.

**What works:** throw out comments shorter than 15 words, then run KMeans asking
for 15 groups. Done separately for each corpus.

**What does not work:** HDBSCAN (a smarter grouping method that turned out to
discard almost everything), and clustering inside a single thread. Section 5.

**Before running this,** you need the embedding cache in
`data/local/embeddings/`. It is too big for Git, so get it from the team Drive
or regenerate it (~4 min on Apple Silicon, ~40 on an older CPU):

```bash
uv run python artifacts/embeddings.py --model BAAI/bge-small-en-v1.5
```


# Summary

### What to use

**KMeans with k=15, on comments of 15 words or longer.** Apply it per corpus,
not once globally — the length filter removes 35% of Coarse Discourse but only
10% of CGA-CMV, because CMV comments are long by nature. Results are saved to
`data/local/clusters_<corpus>.csv`.

| corpus | comments sorted | themes found | themes a reader could name |
|---|---:|---:|---:|
| Coarse Discourse | 72,541 of 112,236 | 15 | 13 |
| CGA-CMV | 37,687 of 41,772 | 15 | 15 |

**Coarse Discourse themes:** PC building · mobile and sysadmin support · game
strategy · game reviews · media and fiction · music creation · sports · travel ·
finance and legal · relationships · fitness and diet · debate and belief · hobby
gear. Two groups stayed vague — general chat, and people explaining how Reddit
works.

**CGA-CMV themes:** veganism · gender identity · sexual consent · US politics ·
guns · abortion · economic inequality · race · feminism · religion · crime and
punishment · immigration, plus three groups about *how people argue* rather than
what they argue about.

### Three findings

**1. Short comments have to be removed first, and no metric warns you.**

Run the clustering on everything and 4 of the 10 groups are not themes at all —
they are just short comments bunched together. One group is literally
`"Thanks. :)"` repeated 8,079 times.

Why: the model turns each comment into numbers that capture *what it is about*.
But a 2-word comment has almost no subject in it, so "Thanks!" about a graphics
card and "Thanks!" about a breakup end up looking identical. Length takes over
from topic.

Removing short comments fixes it completely. The quality score barely moves
(0.018 to 0.022), which is the first sign that the score is not measuring what
matters here.

**2. A near-accidental proof that the themes are real.**

CGA-CMV was built by researchers who hand-matched conversations into **pairs on
the same topic** — one that turned hostile, one that stayed civil.

The clustering never sees that pairing. It has no idea which conversations are
partners. Yet **59.4% of pairs end up in the same theme.** If the themes were
meaningless, you would expect about **6.7%** (1 in 15, pure chance).

In plain terms: humans sorted these conversations by topic, the model sorted them
independently, and the two agree most of the time. In a task with no answer key,
this is the closest thing to one.

**3. What a conversation is about does not predict whether it turns hostile.**

Across CGA-CMV, 52.7% of conversations derail. Broken down by theme, every single
one lands between 49% and 56% — essentially the same rate everywhere.

Gun control and veganism go bad just as often as each other. So whatever causes a
debate to collapse, it is not the subject matter. Useful for the
conversation-quality model in PR #18, and it matches the Task #4b finding that
thread shape alone does not predict derailment either.

### Why the quality score is reported but not trusted

The silhouette score rates how tight and separated the groups are, from -1 to 1.
It is the standard metric when there are no labels to check against. Here it
disagreed with actually reading the comments **five times**, and lost every time:

| situation | the score said | reading the comments said |
|---|---|---|
| HDBSCAN keeping only 3.4% of comments | 0.329 — best in the notebook | useless, it threw away 96.6% |
| removing short comments | 0.018 → 0.022, basically no change | the single biggest improvement made |
| choosing between 5 and 25 themes | all the same (spread of 0.0016) | 15 finds 4 themes that 10 hides |
| clustering inside threads | 0.104-0.192, the highest scores here | the splits are meaningless |
| choosing k on CGA-CMV | 10 is best | 15 reads better and matches the other corpus |

The reason is simple: the score rewards groups that sit in tight, well-separated
balls. Real themes overlap — "game strategy" and "game reviews" are both about
games, so they sit near each other. Tightness is the wrong question.

**So: coverage (how many comments got sorted) and reading the comments decide.
The score is reported for completeness.**

### Limitations

- "Nameable" is one person's judgement on 4-6 comments per group, not a formal
  labelled evaluation.
- k was only tested at 5, 10, 15, 20 and 25. Nothing above 25 was tried.
- The 15-word cutoff itself was never swept — it was chosen because it sits just
  below the median comment length.
- UMAP was not run — it was only needed to rescue HDBSCAN, which KMeans had
  already outperformed, and it would add a team-wide dependency. So the claim
  here is "density clustering fails with *linear* dimensionality reduction",
  not that it fails outright. Reasoning in 5a.
- Within-thread clustering has nothing to check against and is exploratory.


## 0. Setup

In [1]:
import os
from pathlib import Path
import sys

import numpy as np
import pandas as pd

current = Path.cwd().resolve()
REPO_ROOT = next(p for p in [current, *current.parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO_ROOT / "artifacts"))

# clustering.py and embeddings.py resolve data/local/... relative to the working
# directory, and Jupyter starts in notebooks/. Run from the repo root instead.
os.chdir(REPO_ROOT)

from clustering import (
    DEFAULT_MODEL, attach_text, cluster_corpus, cluster_summary, cluster_thread,
    filter_by_length, kmeans_baseline, load_corpus_vectors, reduce_dimensions,
    representative_examples, score_clusters, thread_inputs,
)
from embeddings import CORPORA, load_text_table
from standardize_convokit import iter_standardized

pd.set_option("display.max_colwidth", 140)

MIN_WORDS = 15
K = 15

## 1. Load, and a way to read clusters

`load_corpus_vectors` loads the pre-computed numbers for each comment and drops
empty and deleted ones automatically.

`read_clusters` prints the comments sitting closest to the centre of each group —
the most typical members. **This is how a clustering gets judged here.** There
are no topic labels in the data, so there is no accuracy to calculate; the test
is whether a human can read a group and give it a name.


In [2]:
data = load_corpus_vectors("coarse-discourse", model_name=DEFAULT_MODEL)
text_table = load_text_table("coarse-discourse")

print(f"{len(data):,} usable comments, {data.vectors.shape[1]} dimensions, "
      f"{data.table['conversation_id'].nunique():,} conversations")


def read_clusters(data, labels, texts, facet=None, names=None, n_per_cluster=4, width=150):
    """Print the most central comments in each cluster.

    `names` is the name given to each cluster after reading this output — the
    clustering produces numbers only, so the names are a human judgement
    recorded here, not something the model returned.

    `facet` is an optional per-row Series (subreddit, derailed/civil) summarised
    beside each cluster. For Coarse Discourse it guards against a "theme" that is
    really just one community.
    """
    examples = attach_text(representative_examples(data, labels, n_per_cluster=n_per_cluster), texts)
    frame = pd.DataFrame({"cluster": labels})
    if facet is not None:
        frame["facet"] = np.asarray(facet)
    for cluster_id, group in examples.groupby("cluster"):
        size = group["cluster_size"].iloc[0]
        name = f"  {(names or {}).get(cluster_id, '?')}" if names is not None else ""
        note = ""
        if facet is not None:
            top = frame.loc[frame.cluster == cluster_id, "facet"].value_counts(normalize=True).head(3)
            note = "   [" + ", ".join(f"{s} {p:.0%}" for s, p in top.items()) + "]"
        print(f"\n{'='*74}\nCLUSTER {cluster_id}{name}  —  {size:,} comments ({size/len(data):.1%}){note}")
        for text in group["text"]:
            print(f"   - {' '.join(str(text).split())[:width]}")


subreddit_of = {
    conversation.id: conversation.metadata["source_metadata"].get("subreddit")
    for conversation in iter_standardized(CORPORA["coarse-discourse"])
}
subreddits = data.table["conversation_id"].map(subreddit_of)

112,236 usable comments, 384 dimensions, 9,447 conversations


## 2. The problem: short comments are not a theme

Start by running the clustering on every comment, then read what each group
actually contains. Everything else in this notebook follows from what comes back.


In [3]:
raw_labels = kmeans_baseline(data.vectors, n_clusters=10)
print(f"silhouette {score_clusters(data.vectors, raw_labels):.3f}")

# Names assigned by reading the output below. Four are not topics at all.
RAW_NAMES = {
    0: "(not a theme) thanks + follow-up question",
    1: "link sharing and new-user posts",
    2: "PC and tech support",
    3: "sports",
    4: "personal struggles and advice",
    5: "game strategy and mechanics",
    6: "(not a theme) short 'Thanks. :)' replies",
    7: "(not a theme) one-line jokes",
    8: "(not a theme) short snarky replies",
    9: "debate and belief",
}

read_clusters(data, raw_labels, text_table, facet=subreddits, names=RAW_NAMES)

silhouette 0.018

CLUSTER 0  (not a theme) thanks + follow-up question  —  13,995 comments (12.5%)   [electronic_cigarette 4%, personalfinance 3%, guns 3%]
   - Thank you! That was what I was thinking from the get-go, but wanted to check and see if anyone had the same experience. Thanks for the info!
   - Thanks for the info :) Seems like I'm stuck saving up for a long time though, as I currently lack the time to pull off any of your options^^
   - Awesome information! Thanks for the help, I will take a look into this.
   - Awesome! Thanks so much for the concise reply!

CLUSTER 1  link sharing and new-user posts  —  12,532 comments (11.2%)   [askreddit 4%, movies 2%, anime 2%]
   - Yeah, sorry. I'm new here. I'll be sure to format it better in future installments. Glad you liked it!
   - awesome I love this! Send me link? I'd love to check it out :-)
   - [Ayeee](https://www.youtube.com/watch?v=vdn3zoToISk)
   - Thanks for the suggestions! I'm a bit new to posting on reddit, though I'

**Result: only 5 of the 10 groups are real themes. 4 are just short comments.**

The real ones: PC and tech support, sports, personal advice, game strategy,
debate.

The four marked `(not a theme)` above are groups of short comments: people
saying thanks (8,079 of them), one-line jokes, snarky replies, and
"thanks, and one more question".

**Why this happens.** Each comment was converted into a list of numbers
describing what it is about. A very short comment has barely any subject in it,
so what is left is its *form* — short, polite, jokey. "Thanks!" about a graphics
card and "Thanks!" about a breakup come out nearly identical.

So the clustering is really splitting on two different things at once: what a
comment is about, and how long it is. Those two cut across each other, which is
why nothing forms a clean group and the score sits at 0.018.

Worth noting: group 6 is the **appreciation** category and group 7 is **humor** —
both are discourse acts, which the toolkit already detects with the classifier
from PR #21. The clustering is rediscovering something the team can already do,
instead of finding topics.

**One failure mode ruled out.** No group draws more than 11% of its comments from
any single subreddit. So these are not simply communities in disguise — the
clustering is not just rediscovering r/buildapc.

**Where this fix came from.** Removing short comments is not something the
metrics suggested, and it is not in PR #17's recommendations. It came from
reading the clusters — the silhouette registers almost none of it (0.018 to
0.022). The lesson carried through the rest of the notebook: read the groups
before trusting any number about them.


## 3. The method that works

Two steps: remove the short comments, then decide how many themes to ask for.

The 15-word cutoff matches the `word_count` definition in `text_features.py`
(Task #4a), so it lines up with that notebook's numbers — the median comment
there is 22 words, so this keeps a bit more than half the corpus.

How many themes to ask for (`k`) is a free choice, so it has to be tested rather
than assumed. The table below tries five values.


In [4]:
filtered = filter_by_length(data, text_table, min_words=MIN_WORDS)
print(f"{len(data):,} -> {len(filtered):,} comments "
      f"({len(filtered)/len(data):.1%} kept, {len(data)-len(filtered):,} dropped)")

k_rows, k_labels = [], {}
for k in [5, 10, 15, 20, 25]:
    candidate = kmeans_baseline(filtered.vectors, n_clusters=k)
    sizes = pd.Series(candidate).value_counts()
    k_labels[k] = candidate
    k_rows.append({
        "k": k,
        "silhouette": round(score_clusters(filtered.vectors, candidate), 4),
        "smallest_cluster": int(sizes.min()),
        "largest_share": round(sizes.max() / len(filtered), 3),
    })

pd.DataFrame(k_rows)

112,236 -> 72,541 comments (64.6% kept, 39,695 dropped)


,k,silhouette,smallest_cluster,largest_share
0,5,0.0232,5126,0.285
1,10,0.0220,4174,0.128
2,15,0.0236,2476,0.098
3,20,0.0228,2242,0.079
4,25,0.0224,1886,0.056


**The score gives no guidance at all on how many themes to use.**

Every value from 5 to 25 scores between 0.0220 and 0.0236 — a spread of 0.0016,
which is noise. The metric genuinely cannot tell whether 5 themes or 25 themes is
the better description of this data.

k=15 does happen to score highest, but by a margin far too small to mean
anything. **That is not the reason it was chosen** — the reason is in the next
cell, where the groups are read.


In [5]:
labels = k_labels[K]
filtered_subreddits = filtered.table["conversation_id"].map(subreddit_of)

# Names assigned by reading the output below. Two could not be named.
COARSE_THEMES = {
    0: "PC building",
    1: "(vague) general chat",
    2: "game reviews and purchases",
    3: "relationships",
    4: "(vague) how to use Reddit",
    5: "media, fiction, creative work",
    6: "mobile and sysadmin support",
    7: "hobby gear",
    8: "sports",
    9: "travel and local recommendations",
    10: "finance, career, legal",
    11: "debate and belief",
    12: "fitness and diet",
    13: "game strategy and mechanics",
    14: "music creation and critique",
}

read_clusters(filtered, labels, text_table, facet=filtered_subreddits, names=COARSE_THEMES)


CLUSTER 0  PC building  —  2,733 comments (3.8%)   [buildapc 32%, pcmasterrace 12%, techsupport 7%]
   - The sad thing is that it IS a desktop. Thanks for the help though, I am going to see if I could pop the thing open and check if it has an integrated G
   - Thank you. I'm just now looking at the Lan Gear "Da Box" which seems designed for room for a dual GPU.
   - I'm on mobile so I can't help link but your cooler and storage is something you should change out. You don't actually even need a cooler if you aren't
   - Just going to toss my opinion in here that everything looks good. Maybe more price effective RAM, look for a low CAS Latency number like 7 or 8 and br

CLUSTER 1  (vague) general chat  —  6,065 comments (8.4%)   [askreddit 9%, leagueoflegends 1%, trees 1%]
   - Well, that's a pretty cool story. If you've handled it so far I'd say don't worry about it :)
   - I'm imagining a situation similar to freaky Friday... You might be on to something /s
   - That's crazy, I seriou

**All 15 groups are topics now, and 13 can be given a clear name.**

The names above are a human judgement, recorded in `COARSE_THEMES` — the model
returns numbered groups and nothing else. Two could not be named: cluster 1 is
general chatter and cluster 4 is people explaining how Reddit works.

The subreddit mix in brackets is a sanity check. A group that was 90% one
subreddit would be suspicious — the model would be finding the *community*
rather than the topic. The highest here is PC building at 32% buildapc, so the
groups genuinely cut across communities.

**Why 15 instead of 10.** Two concrete improvements:

- The PC group gets cleaner: buildapc rises from 13% of it to **32%**, because
  phone and sysadmin questions move out into their own group instead of being
  mixed in.
- **Four themes appear that k=10 hides entirely:** travel, fitness and diet,
  music creation, and relationships as distinct from general life advice. At
  k=10, "I'm struggling with my job" and "should I leave my partner" sat
  together.

k=20 was also read. Its extra groups re-split mobile tech rather than finding
anything new, so 15 is the stopping point.


## 4. Second corpus: CGA-CMV

Everything so far used one dataset. This section runs the same method on the
second one to check it was not tuned to a single corpus.

CGA-CMV is ChangeMyView — people posting an opinion and inviting others to argue
them out of it. It was collected to study conversations turning hostile, not to
study topics, so it is a genuinely different test.

It also offers something rare here: the corpus is built from conversation
**pairs that its authors matched on topic by hand**. The clustering knows nothing
about that pairing, so checking whether pairs land together is close to having an
answer key.


In [6]:
cmv = load_corpus_vectors("cga-cmv", model_name=DEFAULT_MODEL)
cmv_text = load_text_table("cga-cmv")
cmv_filtered = filter_by_length(cmv, cmv_text, min_words=MIN_WORDS)

print(f"{len(cmv):,} -> {len(cmv_filtered):,} comments "
      f"({len(cmv_filtered)/len(cmv):.1%} kept)")
print(f"Coarse Discourse kept {len(filtered)/len(data):.1%} for comparison")

cmv_labels = kmeans_baseline(cmv_filtered.vectors, n_clusters=K)
print(f"\nsilhouette {score_clusters(cmv_filtered.vectors, cmv_labels):.4f}")

41,772 -> 37,687 comments (90.2% kept)
Coarse Discourse kept 64.6% for comparison



silhouette 0.0285


**The same filter behaves completely differently here: it removes 9.8% of
comments, against 35% on Coarse Discourse.**

People write long comments when they are arguing a position, so far fewer fall
below 15 words. This is why the filter is applied per corpus rather than being
hard-coded once — the right cutoff depends on how people write in that community.


In [7]:
# CGA-CMV has no subreddit; the useful facet is whether the conversation derailed.
source_to_id, pair_source, derailed_of = {}, {}, {}
for conversation in iter_standardized(CORPORA["cga-cmv"]):
    source_metadata = conversation.metadata["source_metadata"]
    source_to_id[conversation.source_id] = conversation.id
    pair_source[conversation.id] = source_metadata.get("pair_id")
    derailed_of[conversation.id] = source_metadata.get("has_removed_comment")

cmv_derailed = cmv_filtered.table["conversation_id"].map(derailed_of)

# Names assigned by reading the output below. Three are about *how* people argue
# rather than about a subject.
CMV_THEMES = {
    0: "veganism and diet ethics",
    1: "gender identity",
    2: "sexual consent",
    3: "US politics",
    4: "guns",
    5: "(how people argue) burden of proof",
    6: "abortion and bodily autonomy",
    7: "economic inequality",
    8: "media and political discourse",
    9: "(how people argue) CMV rules and persuasion",
    10: "crime and punishment",
    11: "immigration and nationalism",
    12: "feminism",
    13: "race",
    14: "religion",
}

read_clusters(cmv_filtered, cmv_labels, cmv_text,
              facet=cmv_derailed.map({True: "derailed", False: "civil"}),
              names=CMV_THEMES)


CLUSTER 0  veganism and diet ethics  —  2,199 comments (5.8%)   [derailed 52%, civil 48%]
   - >For your first objection, unlike what my post explicitly said not to do, you did: you got hung up on the analogy and did not address the ideas the an
   - Vegans do it for 1. health 2. the environment 3. people working in these terrible conditions 4. animal welfare. People do it for a lot of different re
   - I think your statement "we've evolved for eating animals" is debatable, simply because it has been proven that health is better without eating animals
   - > I just don't think that's true. Again, go ask folks in r/vegan. You're suggesting I go and ask the very people I am accusing of not having good info

CLUSTER 1  gender identity  —  1,993 comments (5.3%)   [derailed 54%, civil 46%]
   - Well, my point was that it just appears you're a bit uneducated about how sex and gender functions together. What does it mean when you say someone is
   - Just because you don't believe in it doesn

**All 15 groups are nameable here — better than the 13 of 15 on Coarse
Discourse.**

Three of them are about *how people argue* rather than a subject: who has to
prove what, what counts as convincing, and the forum's own rules. Those are
still clear groups — just a different kind of category, worth knowing if the
report stage expects every theme to be a subject.

The bracket shows the derailed/civil split per group, which section 4b uses.


### 4a. Do matched pairs land together?

The strongest check available in this whole task.

The researchers who built CGA-CMV took each conversation that turned hostile and
hand-picked a partner conversation **about the same topic** that stayed civil.
The clustering has no access to that pairing.

So: if the themes are real, partners should usually land in the same theme. If
the themes are noise, partners should land together about as often as any two
random conversations would.


In [8]:
by_conversation = pd.DataFrame({
    "conversation_id": cmv_filtered.table["conversation_id"],
    "cluster": cmv_labels,
})
dominant = by_conversation.groupby("conversation_id")["cluster"].agg(
    lambda s: s.value_counts().index[0]
)

same = total = 0
for conversation_id, cluster in dominant.items():
    partner = source_to_id.get(pair_source.get(conversation_id))
    if partner in dominant.index:
        total += 1
        same += int(cluster == dominant[partner])

print(f"matched pairs sharing a dominant cluster: {same:,}/{total:,} = {same/total:.1%}")
print(f"chance if clusters were unrelated to pairing: ~{1/K:.1%}")

matched pairs sharing a dominant cluster: 4,030/6,788 = 59.4%
chance if clusters were unrelated to pairing: ~6.7%


**59.4% of pairs landed in the same theme. Pure chance would be about 6.7%
(1 in 15).**

That is roughly nine times better than random. Humans sorted these conversations
by topic; the model sorted them independently and agreed most of the time.

In an unsupervised task there is normally nothing to check your answer against.
This is the closest thing to a verification in the notebook, and it says the
groups are tracking real topics rather than some artefact of the data.

The 40% that disagree are not necessarily errors — a conversation can sit
genuinely between two themes, and KMeans forces every comment into exactly one.


### 4b. Does topic predict derailment?

A free question now that the themes exist, and directly useful to the
conversation-quality work in PR #18: do some subjects turn hostile more than
others?


In [9]:
derailment = pd.DataFrame({"cluster": cmv_labels, "derailed": np.asarray(cmv_derailed)})
per_cluster = derailment.groupby("cluster")["derailed"].mean().sort_values()

print(f"overall derailed share: {derailment['derailed'].mean():.1%}")
print(f"per-cluster range: {per_cluster.min():.0%} to {per_cluster.max():.0%}")
per_cluster.round(3)

overall derailed share: 52.7%
per-cluster range: 49% to 56%


cluster
6     0.489
5     0.501
14    0.508
7     0.511
10    0.512
3     0.513
0     0.516
11    0.518
8     0.536
12    0.538
1     0.538
4     0.541
13    0.549
2     0.552
9     0.560
Name: derailed, dtype: float64

**No. Topic has essentially nothing to do with it.**

52.7% of CGA-CMV conversations derail overall. Broken down by theme, the range is
**49% to 56%** — every theme sits within a few points of the average.

Concretely: a debate about gun control is no more likely to turn hostile than one
about veganism. The intuitive guess — that inflammatory subjects cause
inflammatory conversations — does not hold in this data.

So whatever drives derailment lives somewhere else: how people argue, the shape
of the thread, who is involved. This matches the Task #4b finding that thread
structure alone does not separate derailed from civil conversations either.


## 5. What did not work

Both approaches below were tested properly and neither is usable. Kept short, but
kept — the evidence is here so nobody repeats them.

### 5a. HDBSCAN (a smarter grouping method)

**Why this was tried first.** PR #17 compared three sentence-embedding models,
found ~0.02 silhouette with all of them, and concluded the bottleneck was the
clustering method rather than the encoder: *"clustering will need a better setup
(e.g. UMAP + HDBSCAN) rather than a different encoder."* This notebook takes
that recommendation as its starting point — the encoder is held fixed at
`bge-small-en-v1.5` throughout and only the method varies.

The first half of that diagnosis holds up: swapping encoders was never
attempted here and nothing suggests it would have helped. The suggested
replacement is the part that did not survive testing.

HDBSCAN should be better than KMeans in principle. KMeans forces every comment
into a group and makes you pick the number of groups in advance. HDBSCAN works
out the number itself and is allowed to say "this comment does not belong
anywhere."

The problem turned out to be that it says that about almost everything.

**The column to watch is `coverage`** — what fraction of comments got put in any
group at all.


In [10]:
rng = np.random.default_rng(42)
sample_idx = np.sort(rng.choice(len(filtered.table), size=20_000, replace=False))
X = filtered.vectors[sample_idx]

hdbscan_rows = []
for dims in [10, 15, 50]:
    reduced = reduce_dimensions(X, method="pca", n_components=dims)
    for min_size in [25, 50, 100]:
        hdbscan_labels = cluster_corpus(reduced, min_cluster_size=min_size)
        summary = cluster_summary(hdbscan_labels)
        hdbscan_rows.append({
            "pca_dims": dims,
            "min_cluster_size": min_size,
            "n_clusters": summary["n_clusters"],
            "coverage": round(1 - summary["noise_rate"], 3),
            "silhouette": round(score_clusters(X, hdbscan_labels), 3),
        })

pd.DataFrame(hdbscan_rows)

,pca_dims,min_cluster_size,n_clusters,coverage,silhouette
0,10,25,2,0.184,0.037
1,10,50,2,0.181,0.037
2,10,100,2,0.117,0.043
3,15,25,4,0.079,0.078
4,15,50,3,0.055,0.098
5,15,100,3,0.033,0.132
6,50,25,5,0.071,0.102
7,50,50,3,0.060,0.128
8,50,100,0,0.000,NaN


**The best result put 18.4% of comments into groups — and found only 2 of
them.** One setting put *nothing* anywhere: zero groups, everything labelled as
leftovers.

Look at how coverage and score move in opposite directions. The best-scoring
setting (0.132) sorted just 3.3% of comments. That is the trap: it is easy to get
tidy-looking groups by keeping only the easiest few percent and discarding the
rest. A report cannot be built on 3% of the data.

Because every setting is bad at one or the other, this is not a tuning problem —
the method does not fit this data.

**A hypothesis that was wrong.** The starting theory was that 384 numbers per
comment is too many for HDBSCAN, so squashing them down would fix it. PCA was
tried at 10, 15 and 50 — all behaved the same way. Squashing was not the answer.

### Why UMAP was not tried

UMAP is the usual next step here. It squashes the numbers down like PCA, but it
can follow curved patterns where PCA only finds straight-line ones, and
UMAP + HDBSCAN is the standard pairing for text embeddings. It was not run, for
three reasons.

**1. It was only ever needed to rescue HDBSCAN.** Squashing is not the goal; it
is a workaround for HDBSCAN needing tightly packed points. Once the length
filter made plain KMeans work (section 3), the problem was already solved.

**2. KMeans sets a bar HDBSCAN would still have to clear.** KMeans sorts **100%**
of comments into 15 groups, 13 of them nameable. The best HDBSCAN result here
sorted **18.4%** and found 2 groups. For UMAP to change the recommendation it
would have to take HDBSCAN from a fifth of the corpus to near-complete coverage
*and* produce groups at least as readable. That is a large gap to close with a
different squashing step, given PCA at 10, 15 and 50 dimensions all behaved the
same way.

**3. It adds a dependency for the whole team.** `uv add umap-learn` changes
`pyproject.toml` and `uv.lock`, so everyone re-syncs their environment. Worth it
for a method being adopted; hard to justify for a method already rejected on
coverage.

**What this costs in certainty.** The honest claim from this notebook is
*"density clustering does not work here with linear dimensionality reduction"* —
not *"density clustering does not work here."* UMAP is the missing test for the
stronger statement. It is cheap (~15 minutes) if a future reader wants to close
that gap, and section 5a's sweep can be rerun with `method="umap"` unchanged
otherwise.

### 5b. Clustering inside a single thread

The task asks for themes within single threads as well as across a corpus. It
does not work, and the reason is structural rather than a tuning failure.


In [11]:
thread_rows = []
for variant, source in [("all comments", data), (">=15 words", filtered)]:
    for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
        groups = []
        for _, thread in list(thread_inputs(source, min_utterances=8))[:300]:
            thread_labels = cluster_thread(thread.vectors, distance_threshold=threshold)
            groups.append(len(set(thread_labels)))
        thread_rows.append({
            "variant": variant,
            "threshold": threshold,
            "mean_groups": round(float(np.mean(groups)), 1),
            "one_group_rate": round(float(np.mean([g == 1 for g in groups])), 3),
        })

pd.DataFrame(thread_rows)

,variant,threshold,mean_groups,one_group_rate
0,all comments,0.3,8.4,0.013
1,all comments,0.4,4.5,0.097
2,all comments,0.5,1.8,0.517
3,all comments,0.6,1.0,0.987
4,all comments,0.7,1.0,1.000
5,>=15 words,0.3,4.4,0.100
6,>=15 words,0.4,2.0,0.490
7,>=15 words,0.5,1.1,0.880
8,>=15 words,0.6,1.0,1.000
9,>=15 words,0.7,1.0,1.000


**There is no setting that works.**

The cutoff controls how eager the method is to split a conversation up. Watch
what happens across the range:

- at 0.3 it splits a thread into 8.4 pieces on average — basically every comment
  on its own
- at 0.6, **99% of threads come back as a single undivided group**
- nothing in between behaves sensibly

If real sub-topics existed, there would be a stretch of settings where the number
of groups stayed roughly steady — the method would be following actual
boundaries in the conversation instead of just obeying the cutoff. There is no
such stretch.

Reading the splits confirms it. A 34-comment thread "split" into 33 + 1, where
the lone comment is *"I highly doubt it though."* That is shaving off one
outlier, not finding a sub-theme. There was one genuine success — a 12-comment
thread correctly separated a joke tangent (*"What's the word?" / "Bird."*) — but
it is the exception.

**Why it fails: a thread is already about one thing.** That is what makes it a
thread. These embeddings group by topic, so inside a single conversation every
comment looks nearly the same to the model and there is nothing left to separate.
Corpus scope works for exactly the opposite reason — across 9,447 threads, topics
genuinely differ.

Filtering short comments makes this *worse*, not better: it removes 36% of the
threads big enough to cluster at all, and pushes the single-group rate from 54%
to 89%. A filter that helps with 112,236 comments hurts when a thread has 8.

**What to use instead.** The real structure inside a thread is conversational —
who replies to whom, who agrees, who disagrees. The toolkit already has both
pieces: the reply tree in the standardized schema, and the discourse-act
classifier from PR #21. Themes are a corpus-level idea in this data.


## 6. Experiment log

Everything tried, in order, including what failed. Reading the table:

- **coverage** = what share of comments ended up in any group (100% means nothing
  was discarded)
- **nameable** = how many groups a reader could give a topic name to
- **silhouette** = the tightness score, included for completeness; see the Summary
  for why it is not the deciding evidence

| # | approach | silhouette | coverage | nameable | outcome |
|---|---|---:|---:|---:|---|
| 1 | KMeans k=10, all comments | 0.018 | 100% | 5 / 10 | **Rejected** — 4 groups are short comments, not topics. Reproduces the embedding comparison's ~0.02, confirming the inputs match. |
| 2 | PCA(50) + HDBSCAN(100), all comments | 0.329\* | 3.4% | not read | **Rejected** — scored 18x better than the baseline while discarding 96.6% of comments. Tried because PR #17 recommended UMAP + HDBSCAN. |
| 3 | PCA(10/15/50) x HDBSCAN(25/50/100), all | 0.044-0.209 | 2.9-10.6% | not read | **Rejected** — 9 settings, none usable. Disproved the idea that 384 dimensions was the blocker. |
| 4 | KMeans k=10, >= 15 words | 0.022 | 100% | 9 / 10 | **Superseded by #7** — all groups became topics. The score barely moved; reading decided it. |
| 5 | Within threads, all comments | 0.104-0.170 | 100% | — | **Rejected** — no workable cutoff; 54% single-group at 0.5, 99% at 0.6. |
| 6 | Within threads, >= 15 words | 0.120-0.192 | 100% | — | **Rejected** — worse. Filtering removes 36% of clusterable threads. |
| 7 | **KMeans k=15, >= 15 words (Coarse Discourse)** | 0.0236 | 100% | **13 / 15** | **Selected** — finds travel, fitness, music and relationships, which k=10 merges away. |
| 8 | KMeans k=5 / 20 / 25, filtered | 0.0224-0.0232 | 100% | — | **Rejected** — 5 is too coarse; 20 re-splits a theme rather than finding new ones. |
| 9 | PCA x HDBSCAN, filtered | 0.037-0.132 | 3.3-18.4% | not read | **Rejected** — filtering did not rescue it; one setting found zero groups. |
| 10 | **KMeans k=15, >= 15 words (CGA-CMV)** | 0.0285 | 100% | **15 / 15** | **Selected** — method transfers. 59.4% matched-pair agreement against 6.7% chance. |
| 11 | UMAP + HDBSCAN | — | — | — | **Not run deliberately** — only needed to rescue HDBSCAN, which KMeans beats on coverage 100% vs 18.4%; adds a team-wide dependency. See 5a. |

\* measured after squashing the numbers down, which flatters itself; every other
score is measured on the original 384 numbers.

### Decisions, and the reasoning

**Removed comments under 15 words.** Experiment 1 showed 4 of 10 groups were
comment types rather than topics. 15 sits just below the median comment length of
22 words (Task #4a), keeping 64.6% of Coarse Discourse. The cutoff itself was not
swept — open work.

**Applied the filter per corpus rather than globally.** It costs 35% on Coarse
Discourse, 9.8% on CGA-CMV, and is actively harmful inside threads.

**Chose KMeans over HDBSCAN.** HDBSCAN is the better tool on paper, but across 18
settings it never sorted more than 18.4% of the comments.

**Chose k=15 by reading the groups, not by the score.** The score was flat across
every k tried.

### Hypotheses that turned out wrong

Recorded so nobody retries them:

- *"HDBSCAN finds nothing because 384 numbers per comment is too many — squash
  them."* PCA at 10 gave 10.6% coverage against 3.4% at 50. Better, still
  unusable. Squashing was not the blocker.
- *"A higher score means better themes."* The highest-scoring run in this
  notebook is the least usable one.
- *"Threads contain sub-topics that clustering can find."* Threads are
  single-topic by construction. Their scores are the highest in the notebook while
  their splits are meaningless.


## 7. Save outputs

One row per sorted comment. These join to the other feature tables
(`text_features`, `structural_features`) on `utterance_id`.

Saved to `data/local/`, which is Git-ignored — share through the team Drive.

**Important for whoever uses this:** only comments of 15+ words have a theme.
That means **35% of Coarse Discourse and 10% of CGA-CMV have no theme at all**.
Downstream code must handle a missing theme rather than assume every comment has
one.


In [12]:
for corpus, source, cluster_labels, full in [
    ("coarse-discourse", filtered, labels, data),
    ("cga-cmv", cmv_filtered, cmv_labels, cmv),
]:
    output = source.table[["utterance_id", "conversation_id"]].copy()
    output["cluster"] = cluster_labels
    output["k"] = K
    output["min_words"] = MIN_WORDS

    output_path = REPO_ROOT / "data" / "local" / f"clusters_{corpus}.csv"
    output.to_csv(output_path, index=False)
    print(f"{corpus}: saved {len(output):,} rows to {output_path.name} "
          f"({len(full) - len(output):,} comments have no cluster)")

coarse-discourse: saved 72,541 rows to clusters_coarse-discourse.csv (39,695 comments have no cluster)
cga-cmv: saved 37,687 rows to clusters_cga-cmv.csv (4,085 comments have no cluster)


## 8. How this connects to the rest of the project

**For anyone using the sentence embeddings (PR #17):** these vectors encode
topic for long comments and *form* for short ones. Any task that groups or
compares comments by similarity should filter by length first, or short comments
will match each other regardless of subject. This is not specific to clustering.

**For the conversation-quality model (PR #18):** topic does not predict
derailment. All 15 CGA-CMV themes derail at 49-56% against 52.7% overall, so a
topic feature is unlikely to help that model. Combined with the Task #4b result
that thread structure alone does not separate derailed from civil either, the
signal is probably in *how* people argue rather than what about or how the
thread is shaped.

**For the discourse classifier (PR #21):** unfiltered clustering spontaneously
produced an "appreciation" group and a "humor" group — two of that model's own
labels — because short comments are dominated by form. Nothing to fix; it is
corroboration from an independent direction that short comments carry discourse
signal rather than topic signal.

**For the report stage (Task #8 onward):** themes are a **corpus-level** concept
in this data. Do not expect to break a single thread into sub-topics; section 5b
shows why that fails structurally. For within-thread organisation, use the reply
tree in the standardized schema plus the discourse-act classifier.
